# OSRM (Contraction Hierarchies) - endustri hiz tabani

Yerel OSRM sunucusu gerekir:  
```bash
docker run -t -v $PWD:/data ghcr.io/project-osrm/osrm-backend osrm-extract -p /opt/car.lua /data/turkey.osm.pbf
osrm-partition ... ; osrm-customize ... ; osrm-routed --algorithm mld /data/turkey.osrm
```

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/06_rota
    %pip install -q requests
print('cwd:', os.getcwd())

## 1. Graf + OD

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C
C.CONFIG.update(dict(
    place='Antakya, Hatay, Turkey',   # veya 'Kahramanmaras, Turkey'
    n_od_pairs=120,                    # senaryo: Antakya 126, Maras 35
))
G = C.load_graph()
closures = C.load_closures()
od = C.sample_od_pairs(G, C.CONFIG['n_od_pairs'], C.CONFIG['seed'])
print('dugum/kenar:', G.number_of_nodes(), G.number_of_edges(), ' kapanma:', len(closures))

## 2. OSRM'e sor

In [ ]:
import requests, time, numpy as np
OSRM = 'http://localhost:5000'
def osrm_route(s, t):
    a = (G.nodes[s]['x'], G.nodes[s]['y']); b = (G.nodes[t]['x'], G.nodes[t]['y'])
    t0 = time.perf_counter()
    r = requests.get(f"{OSRM}/route/v1/driving/{a[0]},{a[1]};{b[0]},{b[1]}?overview=false").json()
    dt = time.perf_counter() - t0
    if r.get('code') != 'Ok': return None
    leg = r['routes'][0]
    return dict(travel_time_s=leg['duration'], distance_m=leg['distance'], compute_s=dt, ok=True)
rs = [osrm_route(s, t) for s, t in od]; ok = [x for x in rs if x]
metrics = dict(n_pairs=len(rs), failed_route_rate=1-len(ok)/len(rs),
               accessibility_ratio=len(ok)/len(rs),
               mean_travel_time_s=float(np.mean([x['travel_time_s'] for x in ok])),
               mean_distance_m=float(np.mean([x['distance_m'] for x in ok])),
               mean_compute_s=float(np.mean([x['compute_s'] for x in ok])))
C.save_result('osrm_ch', metrics); metrics